In [ ]:
from torch.nn import *
import torch
import numpy as np

torch.set_default_tensor_type(torch.cuda.FloatTensor)
M = RNN(5, 16)
m = Sequential(Linear(20, 32), Hardswish(1), Linear(32, 2))
o = torch.optim.Adam((*M.parameters(), *m.parameters()))

In [ ]:
from sklearn.datasets import make_classification
import matplotlib.pyplot as plt


def axy():
    a = [
        make_classification(
            256, 4, n_informative=4, n_redundant=0, n_clusters_per_class=1
        )
        for a in range(256)
    ]
    x = torch.tensor(np.stack([a[0] for a in a], 1), dtype=torch.float)
    y = torch.tensor(np.stack([a[1] for a in a], 1), dtype=torch.long)
    return torch.cat((x, y[..., None].float()), 2), x, y


def train(M, a, zero=torch.zeros(1, 256, 16)):
    return M(a)[0]


def fast_los(m, d, x, y, mask=torch.arange(1, 257)):
    k = torch.randint(256, (256,)) % mask
    return functional.cross_entropy(
        m(torch.cat((d, x[k]), 2)).view(-1, 2), y[k].ravel()
    )


def slow_los(m, d, x, y):
    return sum(
        functional.cross_entropy(
            m(
                torch.cat((d[i].broadcast_to((i + 1,) + d.shape[1:]), x[: i + 1]), 2)
            ).view(-1, 2),
            y[: i + 1].ravel(),
        )
        for i in range(len(d))
    )


def slow_acc(m, d, x, y):
    return [
        (
            m(
                torch.cat((d[i].broadcast_to((i + 1,) + d.shape[1:]), x[: i + 1]), 2)
            ).argmax(2)
            == y[: i + 1]
        )
        .half()
        .mean()
        .item()
        for i in range(len(d))
    ]


while 1:
    for i in range(30):
        for i in range(300):
            a, x, y = axy()
            d = train(M, a)
            i = fast_los(m, d, x, y)
            i.backward()
            o.step()
            o.zero_grad()
        print(i)
    plt.figure()
    plt.plot(slow_acc(m, d, x, y))
    plt.show()